# Chapter 7: Linear Models for Regression

Notebook này là walkthrough ngắn dành cho người mới học. Hãy chạy lần lượt từng cell và quan sát kết quả thay vì cố nhớ mọi công thức.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.data import make_linear_dataset, make_nonlinear_dataset, make_sparse_dataset
from src.evaluation import evaluate_regression
from src.models import create_model, get_estimator, train_model, predict_model
from src.visualization import plot_coefficients, plot_predictions_1d

## Bước 1: Regression là gì?

Regression là bài toán dự đoán một giá trị liên tục, ví dụ giá nhà, doanh thu hoặc mức tiến triển bệnh. Khác với classification, kết quả không phải một nhãn như `spam` hoặc `không spam`.

## Bước 2: Linear Regression và Least Squares

Linear Regression tìm một đường thẳng phù hợp dữ liệu. Least Squares chọn đường làm cho tổng bình phương sai số nhỏ nhất. Bình phương khiến sai số lớn bị phạt mạnh hơn.

In [ ]:
X, y = make_linear_dataset()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
linear = train_model(create_model('Linear Regression'), X_train, y_train)
figure = plot_predictions_1d(linear, X_train, y_train, X_test, y_test, 'Linear Regression')
plt.show()

## Bước 3: Vì sao dùng train/test split?

Tập train dùng để model học. Tập test mô phỏng dữ liệu mới mà model chưa thấy. Nếu chỉ đo trên tập train, ta dễ đánh giá model quá lạc quan.

## Bước 4: Đánh giá bằng MSE, RMSE, MAE và R2

- **MSE:** trung bình bình phương sai số.
- **RMSE:** căn bậc hai của MSE, cùng đơn vị với target.
- **MAE:** trung bình trị tuyệt đối sai số, dễ diễn giải.
- **R2:** mức độ model giải thích biến thiên của target. Càng gần `1` càng tốt.

In [ ]:
predictions = predict_model(linear, X_test)
evaluate_regression(y_test, predictions)

## Bước 5: Ridge xử lý overfitting thế nào?

Ridge thêm hình phạt khi hệ số quá lớn. Ý tưởng đơn giản: model vẫn cần dự đoán tốt nhưng không nên phụ thuộc cực đoan vào một feature. `alpha` càng lớn thì regularization càng mạnh.

In [ ]:
ridge = train_model(create_model('Ridge', alpha=10.0), X_train, y_train)
evaluate_regression(y_test, predict_model(ridge, X_test))

## Bước 6: Lasso chọn biến thế nào?

Lasso dùng regularization L1. Điểm đặc biệt là một số hệ số có thể trở thành đúng `0`. Feature tương ứng gần như bị bỏ khỏi model.

In [ ]:
X_sparse, y_sparse = make_sparse_dataset()
Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_sparse, y_sparse, test_size=0.25, random_state=42)
lasso = train_model(create_model('Lasso', alpha=1.0), Xs_train, ys_train)
figure = plot_coefficients(lasso, list(X_sparse.columns), 'Lasso feature selection')
plt.show()
print('Số hệ số bằng 0:', (get_estimator(lasso).coef_ == 0).sum())

## Bước 7: Kernel Ridge và SVR xử lý dữ liệu phi tuyến thế nào?

Một đường thẳng không thể mô tả tốt dữ liệu dạng sin. Kernel giúp model học quan hệ cong mà không cần tự tạo hàng loạt feature phức tạp.

In [ ]:
X_curve, y_curve = make_nonlinear_dataset()
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_curve, y_curve, test_size=0.25, random_state=42)
for name, model in {
    'Linear Regression': create_model('Linear Regression'),
    'Kernel Ridge': create_model('Kernel Ridge', alpha=0.1, gamma=5.0),
    'SVR': create_model('SVR', C=10.0, gamma=5.0),
}.items():
    train_model(model, Xc_train, yc_train)
    print(name, evaluate_regression(yc_test, predict_model(model, Xc_test)))

## Bước 8: Tổng kết khi nào dùng model nào?

| Model | Khi nên thử |
| --- | --- |
| Linear Regression | Baseline đầu tiên, dễ giải thích |
| Ridge | Nhiều feature liên quan nhau hoặc cần giảm overfitting |
| Lasso | Muốn model gọn hơn và cần feature selection |
| Kernel Ridge | Quan hệ phi tuyến, dataset vừa hoặc nhỏ |
| SVR | Quan hệ phi tuyến, muốn kiểm soát epsilon margin |

Không có model tốt nhất cho mọi dataset. Luôn giữ một tập test và so sánh bằng metric phù hợp.